## 模块 6：高价值客户订单明细查询（EXISTS 子查询）
### 业务需求
筛选名下存在单笔订单营收≥5000元的客户，输出该客户所有历史订单、门店、商品、会员信息，用于高消费潜力客户分层运营

### 思路分析
1. 使用EXISTS做存在性匹配：只要客户任意一笔订单金额达到5000，该客户全部订单都会被筛选出来；
2. 仅做行级条件判断，无多层聚合嵌套查询，执行逻辑简单清晰；
3. 采用星型模型四表关联，带出门店、商品、会员等业务可读维度；
4. 订单日期倒序展示，优先展示近期消费记录

### 核心知识点
1. EXISTS 半连接存在性查询，短路扫描机制；
2. 零售星型模型多表INNER JOIN关联；
3. 订单指标实时计算、数值条件过滤

## SQL 代码

In [1]:
import pandas as pd
import sqlite3

# 1. 连接SQLite数据库
conn = sqlite3.connect("/kaggle/input/datasets/tclaim/retail-sales/retail_sales.db")

# 4. 封装通用SQL执行函数，自动打印表格结果
def query_sql(sql):
    return pd.read_sql(sql, conn)
    
print("数据导入完成，调用 query_sql('SQL语句') 即可执行查询")

数据导入完成，调用 query_sql('SQL语句') 即可执行查询


In [8]:
sql = '''
SELECT
    o.订单ID,
    o.销售日期,
    s.门店名称,
    p.商品类别,
    c.会员等级,
    o.销售数量 * o."单价(元)" AS 订单营收
FROM "销售订单表" o
INNER JOIN "门店信息表" s 
    ON o.门店ID = s.门店ID
INNER JOIN "商品信息表" p 
    ON o.商品ID = p.商品ID
INNER JOIN "客户信息表" c 
    ON o.客户ID = c.客户ID
-- 判断该客户是否产生过单笔5000元及以上的大额订单
WHERE EXISTS (
    SELECT 1
    FROM "销售订单表" o2
    WHERE o2.客户ID = o.客户ID
    AND o2.销售数量 * o2."单价(元)" >= 5000
)
ORDER BY o.客户ID, o.销售日期 DESC;
'''
query_sql(sql)

,订单ID,销售日期,门店名称,商品类别,会员等级,订单营收
0,OD0247,2025-02-10 00:00:00,黄冈门店106,美妆个护,金卡会员,8379.0
1,OD0049,2025-01-30 00:00:00,衡水门店297,母婴用品,普通会员,11578.0
2,OD0279,2025-01-12 00:00:00,呼和浩特门店322,服饰鞋包,普通会员,11109.0
3,OD0114,2025-05-21 00:00:00,德宏傣族景颇族自治州门店154,美妆个护,普通会员,15174.0
4,OD0310,2025-02-07 00:00:00,厦门门店245,服饰鞋包,钻石会员,46942.5
...,...,...,...,...,...,...
290,OD0192,2025-03-20 00:00:00,赣州门店202,母婴用品,金卡会员,3784.0
291,OD0220,2025-02-07 00:00:00,白城门店141,数码家电,金卡会员,37278.0
292,OD0083,2025-05-02 00:00:00,楚雄彝族自治州门店179,母婴用品,普通会员,26439.0
293,OD0056,2025-04-12 00:00:00,咸阳门店384,数码家电,普通会员,142902.0


### 结果说明
1. 输出内容：曾经产生过大额订单客户的全部历史订单，包含客户ID、门店、品类、会员等级多维度信息；
2. 排序优势：同一客户的所有订单聚合在一起，内部由近到远展示消费记录，便于连贯分析单个客户消费习惯；
3. 业务价值：定位高潜力消费客群，分析其全周期消费习惯，针对性推出高端商品专属活动；
4. 代码优势：EXISTS采用短路扫描，匹配到符合条件的订单后即刻停止该行数据扫描，查询执行效率更高